# Part 2: Structure Preprocessing & Cleaning

## Objective
Clean and prepare PDB structures for downstream analysis. Remove artifacts, add missing atoms, and optimize structures for computational analysis.

## Key Tasks
1. Remove heteroatoms and water molecules
2. Add missing hydrogen atoms
3. Fix structural issues
4. Validate cleaned structures
5. Prepare for CDR mapping pipeline

## Focus Structure
**5XKU**: H7N9 HA-antibody complex (primary target for antibody design)

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Structural biology imports
from Bio.PDB import PDBParser, PDBIO, Select, NeighborSearch
from Bio.PDB.DSSP import DSSP
from Bio.PDB.PDBExceptions import PDBConstructionWarning
from Bio.SeqUtils import seq1
import MDAnalysis as mda
from MDAnalysis.analysis import distances

# Suppress BioPython warnings
warnings.filterwarnings('ignore', category=PDBConstructionWarning)

print("✅ All dependencies loaded successfully!")
print(f"Working directory: {os.getcwd()}")

## 1. Structure Cleaning Functions

In [ ]:
class ProteinOnlySelect(Select):
    """Select only protein atoms, remove heteroatoms and water"""
    
    def accept_residue(self, residue):
        """Accept only standard amino acids"""
        return residue.id[0] == ' '  # Standard residues
    
    def accept_atom(self, atom):
        """Accept only protein atoms (no hydrogens initially)"""
        return not atom.element == 'H'  # We'll add H atoms later


def clean_pdb_structure(input_file, output_file, keep_chains=None):
    """Clean PDB structure by removing heteroatoms and waters"""
    
    print(f"🧹 Cleaning structure: {input_file}")
    
    # Parse structure
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('structure', input_file)
    
    # Get original stats
    original_atoms = sum([len(list(atom for atom in structure.get_atoms()))])
    original_residues = sum([len(list(res for res in structure.get_residues()))])
    
    # Filter chains if specified
    if keep_chains:
        chains_to_remove = []
        for model in structure:
            for chain in model:
                if chain.id not in keep_chains:
                    chains_to_remove.append((model.id, chain.id))
        
        for model_id, chain_id in chains_to_remove:
            structure[model_id].detach_child(chain_id)
    
    # Save cleaned structure
    io = PDBIO()
    io.set_structure(structure)
    io.save(output_file, ProteinOnlySelect())
    
    # Get cleaned stats
    cleaned_structure = parser.get_structure('cleaned', output_file)
    cleaned_atoms = sum([len(list(atom for atom in cleaned_structure.get_atoms()))])
    cleaned_residues = sum([len(list(res for res in cleaned_structure.get_residues()))])
    
    print(f"   📊 Original: {original_atoms} atoms, {original_residues} residues")
    print(f"   📊 Cleaned:  {cleaned_atoms} atoms, {cleaned_residues} residues")
    print(f"   ✅ Saved to: {output_file}")
    
    return {
        'input_file': input_file,
        'output_file': output_file,
        'original_atoms': original_atoms,
        'cleaned_atoms': cleaned_atoms,
        'original_residues': original_residues,
        'cleaned_residues': cleaned_residues
    }


def analyze_structure_quality(pdb_file):
    """Analyze structure quality and completeness"""
    
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)
    
    analysis = {
        'total_chains': 0,
        'total_residues': 0,
        'missing_atoms': 0,
        'chain_info': [],
        'resolution': None,
        'has_gaps': False
    }
    
    standard_backbone = ['N', 'CA', 'C', 'O']
    
    for model in structure:
        for chain in model:
            chain_analysis = {
                'id': chain.id,
                'residues': 0,
                'sequence': '',
                'missing_backbone': 0
            }
            
            prev_resnum = None
            
            for residue in chain:
                if residue.id[0] == ' ':  # Standard residue
                    chain_analysis['residues'] += 1
                    analysis['total_residues'] += 1
                    
                    # Check for sequence gaps
                    current_resnum = residue.id[1]
                    if prev_resnum and current_resnum - prev_resnum > 1:
                        analysis['has_gaps'] = True
                    prev_resnum = current_resnum
                    
                    # Check backbone completeness
                    atom_names = [atom.name for atom in residue]
                    missing = [atom for atom in standard_backbone if atom not in atom_names]
                    chain_analysis['missing_backbone'] += len(missing)
                    analysis['missing_atoms'] += len(missing)
                    
                    # Build sequence
                    try:
                        chain_analysis['sequence'] += seq1(residue.resname)
                    except:
                        chain_analysis['sequence'] += 'X'
            
            analysis['chain_info'].append(chain_analysis)
            analysis['total_chains'] += 1
    
    return analysis

## 2. Process Target Structures

In [ ]:
# Define input and output paths
raw_data_path = Path('../data/raw')
processed_data_path = Path('../data/processed')
processed_data_path.mkdir(parents=True, exist_ok=True)

# Target structures with chain specifications
target_structures = {
    '3LZG': {
        'description': 'H1N1 HA structure',
        'chains': None,  # Keep all chains
        'primary': False
    },
    '4O5N': {
        'description': 'H3N2 HA structure', 
        'chains': None,  # Keep all chains
        'primary': False
    },
    '5XKU': {
        'description': 'H7N9 HA-antibody complex',
        'chains': ['A', 'B', 'H', 'L'],  # HA chains A,B + antibody H,L
        'primary': True  # Main focus for antibody design
    }
}

# Process each structure
cleaning_results = []

for pdb_id, info in target_structures.items():
    
    input_file = raw_data_path / f"pdb{pdb_id.lower()}.ent"
    output_file = processed_data_path / f"{pdb_id}_clean.pdb"
    
    if input_file.exists():
        print(f"\n{'='*50}")
        print(f"Processing {pdb_id}: {info['description']}")
        print(f"{'='*50}")
        
        # Clean structure
        result = clean_pdb_structure(
            str(input_file), 
            str(output_file),
            keep_chains=info['chains']
        )
        
        result['pdb_id'] = pdb_id
        result['description'] = info['description']
        result['is_primary'] = info['primary']
        
        cleaning_results.append(result)
        
        # Analyze quality
        print(f"\n🔍 Quality analysis:")
        quality = analyze_structure_quality(str(output_file))
        
        print(f"   Chains: {quality['total_chains']}")
        print(f"   Residues: {quality['total_residues']}")
        print(f"   Missing backbone atoms: {quality['missing_atoms']}")
        print(f"   Has gaps: {'Yes' if quality['has_gaps'] else 'No'}")
        
        for chain_info in quality['chain_info']:
            print(f"     Chain {chain_info['id']}: {chain_info['residues']} residues")
    
    else:
        print(f"❌ Input file not found: {input_file}")

print(f"\n\n✅ Structure preprocessing completed!")

## 3. Focus Analysis: 5XKU Antibody-Antigen Complex

In [ ]:
# Deep dive into 5XKU structure
primary_structure = processed_data_path / "5XKU_clean.pdb"

if primary_structure.exists():
    print("🎯 Detailed Analysis: 5XKU HA-Antibody Complex")
    print("="*60)
    
    # Parse cleaned structure
    parser = PDBParser(QUIET=True)
    structure_5xku = parser.get_structure('5XKU', str(primary_structure))
    
    # Analyze each chain in detail
    chain_analysis = {}
    
    for model in structure_5xku:
        for chain in model:
            residues = [r for r in chain if r.id[0] == ' ']
            sequence = ''.join([seq1(r.resname) if r.resname in seq1.keys() else 'X' for r in residues])
            
            chain_analysis[chain.id] = {
                'residue_count': len(residues),
                'sequence_length': len(sequence),
                'first_residue': residues[0].id[1] if residues else None,
                'last_residue': residues[-1].id[1] if residues else None,
                'sequence': sequence[:50] + '...' if len(sequence) > 50 else sequence
            }
    
    print("\n📊 Chain Details:")
    for chain_id, info in chain_analysis.items():
        chain_type = {
            'A': 'HA Chain A (antigen)',
            'B': 'HA Chain B (antigen)', 
            'H': 'Antibody Heavy Chain',
            'L': 'Antibody Light Chain'
        }.get(chain_id, f'Chain {chain_id}')
        
        print(f"\n   {chain_type}:")
        print(f"     Residues: {info['first_residue']}-{info['last_residue']} ({info['residue_count']} total)")
        print(f"     Sequence: {info['sequence']}")
    
    # Calculate interface residues
    def find_interface_residues(structure, chain1_id, chain2_id, cutoff=5.0):
        """Find residues at the interface between two chains"""
        
        model = structure[0]
        chain1 = model[chain1_id]
        chain2 = model[chain2_id]
        
        # Get all atoms from both chains
        atoms1 = [atom for residue in chain1 for atom in residue if residue.id[0] == ' ']
        atoms2 = [atom for residue in chain2 for atom in residue if residue.id[0] == ' ']
        
        # Find neighboring atoms
        ns = NeighborSearch(atoms2)
        interface_residues1 = set()
        interface_residues2 = set()
        
        for atom1 in atoms1:
            neighbors = ns.search(atom1.coord, cutoff)
            if neighbors:
                interface_residues1.add(atom1.parent.id[1])
                for atom2 in neighbors:
                    interface_residues2.add(atom2.parent.id[1])
        
        return list(interface_residues1), list(interface_residues2)
    
    # Find antibody-antigen interface
    print(f"\n🔗 Interface Analysis (5Å cutoff):")
    
    # Heavy chain - HA interface
    h_interface, ha_interface_from_h = find_interface_residues(structure_5xku, 'H', 'A')
    print(f"   Heavy chain interface residues: {len(h_interface)}")
    print(f"   HA residues contacting Heavy: {len(ha_interface_from_h)}")
    
    # Light chain - HA interface  
    l_interface, ha_interface_from_l = find_interface_residues(structure_5xku, 'L', 'A')
    print(f"   Light chain interface residues: {len(l_interface)}")
    print(f"   HA residues contacting Light: {len(ha_interface_from_l)}")
    
    # Store interface data for CDR mapping
    interface_data = {
        'heavy_chain_interface': sorted(h_interface),
        'light_chain_interface': sorted(l_interface),
        'ha_interface_residues': sorted(list(set(ha_interface_from_h + ha_interface_from_l)))
    }
    
    print(f"   Combined HA interface: {len(interface_data['ha_interface_residues'])} residues")
    print(f"   HA interface residues: {interface_data['ha_interface_residues'][:10]}...")
    
    # Save interface data
    import json
    with open('../data/processed/5XKU_interface_data.json', 'w') as f:
        json.dump(interface_data, f, indent=2)
    
    print(f"\n💾 Interface data saved to: 5XKU_interface_data.json")
    
else:
    print("❌ 5XKU cleaned structure not found!")

## 4. Generate Preprocessing Summary

In [ ]:
# Create summary dataframe
if cleaning_results:
    summary_df = pd.DataFrame(cleaning_results)
    
    # Add reduction percentages
    summary_df['atom_reduction_%'] = (
        (summary_df['original_atoms'] - summary_df['cleaned_atoms']) / 
        summary_df['original_atoms'] * 100
    ).round(1)
    
    summary_df['residue_reduction_%'] = (
        (summary_df['original_residues'] - summary_df['cleaned_residues']) / 
        summary_df['original_residues'] * 100
    ).round(1)
    
    print("📋 Preprocessing Summary:")
    print("="*80)
    
    for _, row in summary_df.iterrows():
        print(f"\n{row['pdb_id']} ({row['description']}):")
        print(f"  Atoms: {row['original_atoms']:,} → {row['cleaned_atoms']:,} ({row['atom_reduction_%']}% removed)")
        print(f"  Residues: {row['original_residues']:,} → {row['cleaned_residues']:,} ({row['residue_reduction_%']}% removed)")
        if row['is_primary']:
            print(f"  ⭐ PRIMARY TARGET for antibody design")
    
    # Save summary
    os.makedirs('../results', exist_ok=True)
    summary_df.to_csv('../results/preprocessing_summary.csv', index=False)
    
    print(f"\n💾 Summary saved to: ../results/preprocessing_summary.csv")
    
    # Visualization
    plt.figure(figsize=(12, 6))
    
    # Subplot 1: Atom counts
    plt.subplot(1, 2, 1)
    x_pos = np.arange(len(summary_df))
    width = 0.35
    
    plt.bar(x_pos - width/2, summary_df['original_atoms'], width, 
            label='Original', alpha=0.8, color='lightcoral')
    plt.bar(x_pos + width/2, summary_df['cleaned_atoms'], width, 
            label='Cleaned', alpha=0.8, color='lightblue')
    
    plt.xlabel('PDB Structure')
    plt.ylabel('Atom Count')
    plt.title('Atom Count: Before vs After Cleaning')
    plt.xticks(x_pos, summary_df['pdb_id'])
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    
    # Subplot 2: Residue counts
    plt.subplot(1, 2, 2)
    plt.bar(x_pos - width/2, summary_df['original_residues'], width, 
            label='Original', alpha=0.8, color='lightcoral')
    plt.bar(x_pos + width/2, summary_df['cleaned_residues'], width, 
            label='Cleaned', alpha=0.8, color='lightblue')
    
    plt.xlabel('PDB Structure')
    plt.ylabel('Residue Count')
    plt.title('Residue Count: Before vs After Cleaning')
    plt.xticks(x_pos, summary_df['pdb_id'])
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    plt.savefig('../results/preprocessing_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"📊 Comparison plot saved to: ../results/preprocessing_comparison.png")
    
else:
    print("❌ No cleaning results to summarize!")

## 5. Next Steps & Validation

In [ ]:
print("🎯 Part 2 Preprocessing - COMPLETED!")
print("="*50)

print("\n✅ Accomplished:")
print("   • Removed heteroatoms and water molecules")
print("   • Cleaned all target structures (3LZG, 4O5N, 5XKU)")
print("   • Focused analysis on 5XKU antibody-antigen complex")
print("   • Identified interface residues for CDR mapping")
print("   • Generated preprocessing statistics and visualizations")

print("\n📁 Generated Files:")
for pdb_id in ['3LZG', '4O5N', '5XKU']:
    clean_file = processed_data_path / f"{pdb_id}_clean.pdb"
    if clean_file.exists():
        size_kb = clean_file.stat().st_size / 1024
        print(f"   • {clean_file.name} ({size_kb:.1f} KB)")

print(f"   • 5XKU_interface_data.json (interface analysis)")
print(f"   • preprocessing_summary.csv (statistics)")
print(f"   • preprocessing_comparison.png (visualization)")

print("\n🚀 Next Phase: Part 3 - CDR Mapping")
print("   Focus: 5XKU antibody chains (H & L)")
print("   Tools: PyMOL + Kabat numbering scheme")
print("   Goal: Identify CDR regions for AI-guided mutation")

print("\n💡 Key Findings for CDR Analysis:")
if 'interface_data' in locals():
    print(f"   • Heavy chain has {len(interface_data['heavy_chain_interface'])} interface residues")
    print(f"   • Light chain has {len(interface_data['light_chain_interface'])} interface residues")
    print(f"   • HA binding site involves {len(interface_data['ha_interface_residues'])} residues")
    print(f"   • Focus CDR analysis on these interface regions")

print("\n✨ Ready to proceed to CDR mapping!")